In [1]:
import torch 
from torch import nn

In [ ]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Conv2d(1, 96, stride=4, kernel_size=11, padding=1) # (224-11)+2x1 / 4 + 1= 54x54
        self.l3 = nn.MaxPool2d(kernel_size=3, stride=2) # (54-3)/2 + 1 = 26x26
        self.l4 = nn.ReLU() # 26x26
        self.l5 = nn.MaxPool2d(kernel_size=3, stride=2) # (26-3)/2 + 1 = 12x12
        self.l6 = nn.Conv2d(256, 384, kernel_size=3, padding=1) # (12-3+2)/1 + 1 = 12
        self.l7 = nn.ReLU()
        self.l8 = nn.Conv2d(384, 384, kernel_size=3, padding=1) # 12x12
        self.l9 = nn.ReLU()
        self.l10 = nn.Conv2d(384, 256, kernel_size=3, padding=1) # 12x12
        self.l11 = nn.ReLU()
        self.l12 = nn.MaxPool2d(kernel_size=3, stride=2) # (12-3)/2 + 1 = 5x5
        self.l13 = nn.Flatten() # 256 x 5x5 = 6400
        self.l14 = nn.Linear(6400, 4096)
        self.l15 = nn.ReLU()
        self.l16 = nn.Dropout(p=0.5)
        self.l17 = nn.Linear(4096, 4096)
        self.l18 = nn.ReLU()
        self.l19 = nn.Dropout(p=0.5)
        self.l20 = nn.Linear(4096, 10)

    def forward(self, x):
        x = x.reshape(-1, 1, 224, 224)
        x = self.l1(x)
        x = self.l3(x)
        x = self.l4(x)
        x = self.l5(x)
        x = self.l6(x)
        x = self.l7(x)
        x = self.l8(x)
        x = self.l9(x)
        x = self.l10(x)
        x = self.l11(x)
        x = self.l12(x)
        x = self.l13(x)
        x = self.l14(x)
        x = self.l15(x)
        x = self.l16(x)
        x = self.l17(x)
        x = self.l18(x)
        x = self.l19(x)
        x = self.l20(x)
        return x



In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define the transformation: Convert to tensor and normalize
# Fashion-MNIST is grayscale, so we use mean and std for one channel
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Training data
train_set = datasets.FashionMNIST(
    root="data", 
    train=True, 
    download=True, 
    transform=transform
)

# Test data
test_set = datasets.FashionMNIST(
    root="data", 
    train=False, 
    download=True, 
    transform=transform
)

In [6]:
train_set[0][0].shape

torch.Size([1, 224, 224])

In [7]:
net = AlexNet()
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

In [8]:
epochs = 10
batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='cuda')

In [11]:
net.to(device)

AlexNet(
  (l1): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4), padding=(1, 1))
  (l3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (l4): ReLU()
  (l5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (l6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (l7): ReLU()
  (l8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (l9): ReLU()
  (l10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (l11): ReLU()
  (l12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (l13): Flatten(start_dim=1, end_dim=-1)
  (l14): Linear(in_features=6400, out_features=4096, bias=True)
  (l15): ReLU()
  (l16): Dropout(p=0.5, inplace=False)
  (l17): Linear(in_features=4096, out_features=4096, bias=True)
  (l18): ReLU()
  (l19): Dropout(p=0.5, inplace=False)
  (l20): Linear(in_features=4096, out_features=10, bias=True)
)

In [12]:
for i in range(epochs):
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        y_predicted = net(X)
        l = loss(y_predicted, y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
    
    print(f"Epoch: {epochs+1}, loss: {l.item(): .4f}")

RuntimeError: Given groups=1, weight of size [384, 256, 3, 3], expected input[128, 96, 12, 12] to have 256 channels, but got 96 channels instead